# Get to Know a Dataset: Pan-Primate Reference Genome Project (PanPrimate-T2T)

This notebook serves as a guided tour of the [Pan-Primate Reference Genome Project (PanPrimate-T2T)](https://registry.opendata.aws/panprimate-t2t) dataset. More usage examples, tutorials, and documentation for this dataset and others can be found at the [Registry of Open Data on AWS](https://registry.opendata.aws/).

### Q: How have you organized your dataset? Help us understand the key prefix structure of your S3 bucket.

The bucket has two top-level prefixes:

```
s3://primate-t2t-genomics-open/
├── manifests/
│   └── sample_data_manifest.csv       # one row per released assembly version - the index into everything else
└── species_data/
    └── <accession_id>/                # one directory per sample, e.g. PR00232
        ├── raw_sequencing/            # append-only raw reads (POD5, HiFi/Kinnex BAM), one file per run
        ├── metadata/                  # per-sample metadata JSON (instrument, chemistry, run accessions)
        └── v<major>.<minor>/          # one directory per assembly version (e.g. v1.0, v1.1)
            ├── assembly/              # primary/alternate FASTA + QC
            ├── annotation/            # gene models (primary + alternate), repeat annotation
            ├── variants/              # VCF calls
            ├── alignments/            # reads aligned to this assembly version
            ├── methylation/           # ONT-based methylation calls
            └── fiberseq/              # PacBio Fiber-seq output
```

**Note:** the raw phased haplotype assemblies (`hap1`/`hap2` FASTA) are archived at NCBI rather than duplicated here, alongside raw chromatin capture (Hi-C) reads. The derived primary/alternate assemblies (one representative sequence per chromosome pair, built from hap1/hap2) are hosted here on AWS, along with everything downstream of them - annotations, variant calls, alignments, methylation, and Fiber-seq output - plus the raw signal-level sequencing data (POD5, HiFi/Kinnex BAM) that has no home at NCBI.

A separate `comparative/` prefix holds multi-species alignments (HAL/MAF), which span many samples rather than belonging to one.

Full technical documentation: https://github.com/sudmantlab/PanPrimate-T2T/blob/main/README_dataset.md


In [ ]:
# This notebook requires the following additional libraries
# (please install using the preferred method for your environment, e.g. pip, conda):
#
# boto3 >= 1.38.23
# polars >= 1.30.0
# matplotlib >= 3.10.3
# pysam >= 0.22.0

# Built-ins
import io

# Installed libraries
import boto3, polars, matplotlib.pyplot as plt, pysam
from botocore import UNSIGNED
from botocore.config import Config


Next, we'll define the location of our dataset, create our boto3 S3 client, and list the top-level prefixes in our bucket.

In [ ]:
bucket = "primate-t2t-genomics-open"

# Public bucket - no need to sign requests
s3 = boto3.client('s3', config=Config(signature_version=UNSIGNED))

for item in s3.list_objects_v2(Bucket=bucket, Delimiter='/')['CommonPrefixes']:
    print(item['Prefix'])


Looking inside `species_data/`, each sample gets its own accession-ID-keyed prefix:

In [ ]:
for item in s3.list_objects_v2(Bucket=bucket, Prefix='species_data/', Delimiter='/', MaxKeys=10)['CommonPrefixes']:
    print(item['Prefix'])


### Q: What data formats are present in your dataset? What kinds of data are stored using these formats? Can you give any advice for how you work with these data formats?

Our dataset spans the full sequencing-to-analysis lineage, in several standard bioinformatics formats:

- **POD5** - raw Oxford Nanopore signal data, one file per sequencing run.
- **BAM** - PacBio HiFi/Kinnex reads and all read alignments, indexed with `.bai` for coordinate-based random access.
- **bgzip-compressed FASTA** (`.fa.gz` + `.fai` index) - primary/alternate genome assemblies. (Raw haplotype-resolved `hap1`/`hap2` FASTA is archived at NCBI rather than hosted here - see the note in the previous section.)
- **bgzip-compressed GFF3** (`.gff3.gz` + `.tbi` index) - gene annotation.
- **bgzip-compressed VCF** (`.vcf.gz` + `.tbi` index) - variant calls.
- **MAF/HAL** - pairwise and multi-species whole-genome comparative alignments.
- **CSV/JSON** - the sample manifest and per-sample metadata records.

We chose indexed, bgzip/BAM formats specifically because they support **HTTP byte-range requests directly against S3** - `samtools`, `bcftools`, `tabix`, `pysam`, IGV, and JBrowse can all stream a single genomic interval without downloading the full (often many-gigabyte) file. This is the core reason the dataset is practical to work with directly on AWS.

**AWS services that may be useful when working with this data:**
- **Amazon S3** - all data lives here, directly streamable via byte-range requests.
- **Amazon EC2** - run alignment, variant calling, or comparative genomics workflows in the same region as the data.
- **AWS Batch** - array jobs iterating over the manifest (e.g. one job per sample) for large-scale reprocessing.
- **Amazon SageMaker** - combining raw signal data (ONT POD5, PacBio HiFi) with assemblies and annotations for training genomic foundation models.
- **AWS HealthOmics** - whole-genome alignment and variant discovery workflows using the released BAM/VCF/assembly files as inputs.


### Q: Can you show us an example of downloading and loading data from your dataset?

As an example, let's load the sample manifest - the entry point for navigating every other file in the bucket - directly into a Polars DataFrame.


In [ ]:
obj = s3.get_object(Bucket=bucket, Key="manifests/sample_data_manifest.csv")
manifest = polars.read_csv(io.BytesIO(obj["Body"].read()))
manifest.head()


Let's look at which samples currently have a released, latest assembly:

In [ ]:
released = manifest.filter(
    (polars.col("assembly_status") == "released") & (polars.col("is_latest") == "TRUE")
)
released.select(["genome_id", "common_name", "species_id", "release_date"])


### Q: A picture is worth a thousand words. Show us a visual (or several!) from your dataset that either illustrates something informative about your dataset, or that you think might excite someone to dig in further.

Below, we stream a single genomic interval directly from an indexed, released assembly's aligned BAM file using HTTP byte-range requests - only that interval is fetched, not the full multi-gigabyte file.


In [ ]:
example = released.row(0, named=True)
print(f"Using {example['genome_id']} ({example['common_name']})")

bam_url = f"https://{bucket}.s3.amazonaws.com/{example['alignment_reads']}"
bam = pysam.AlignmentFile(bam_url, "rb")

region_reads = list(bam.fetch("chr7", 1_000_000, 1_050_000))
print(f"Fetched {len(region_reads)} aligned reads overlapping chr7:1,000,000-1,050,000")


In [ ]:
starts = [r.reference_start for r in region_reads]
lengths = [r.query_alignment_length for r in region_reads]

fig, ax = plt.subplots(figsize=(12, 7), dpi=100, facecolor='white')
for i, (s, l) in enumerate(zip(starts, lengths)):
    ax.plot([s, s + l], [i, i], color="#3498db", linewidth=1, alpha=0.8)
ax.set_xlabel("Position (chr7)", fontsize=12, labelpad=10)
ax.set_ylabel("Read index", fontsize=12, labelpad=10)
ax.set_title(f"HiFi reads aligned to {example['common_name']} ({example['genome_id']}), chr7:1,000,000-1,050,000",
             fontsize=14, pad=20, fontweight='bold')
ax.set_facecolor('#f8f9fa')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.show()


### Q: What is one question that you have answered using these data? Can you show us how you came to that answer?

Because these are chromosome-scale, telomere-to-telomere assemblies rather than fragmented drafts, one question we can answer that's often out of reach for standard reference genomes is: **what fraction of the genome is actually repetitive, and which repeat classes dominate?** Fragmented assemblies frequently collapse or omit highly repetitive regions (satellites, centromeric arrays); a T2T assembly resolves them, so this number is a genuinely informative check on assembly completeness as well as biology in its own right.

Below, we stream a sample's `repeat_masker_pri` output directly from S3 and parse it - this is native RepeatMasker `.out` format: a 3-line header followed by whitespace-delimited records (not standard TSV), so we parse it accordingly.


In [ ]:
rm_key = example["repeat_masker_pri"]
obj = s3.get_object(Bucket=bucket, Key=rm_key)
rm_lines = obj["Body"].read().decode("utf-8").splitlines()

records = []
for line in rm_lines[3:]:  # skip 2 header lines + 1 blank line
    if not line.strip():
        continue
    fields = line.split()
    if fields[-1] == "*":
        fields = fields[:-1]
    records.append({
        "query_seq": fields[4],
        "query_begin": int(fields[5]),
        "query_end": int(fields[6]),
        "repeat_class": fields[10],
    })

repeats = polars.DataFrame(records)
repeats = repeats.with_columns(
    (polars.col("query_end") - polars.col("query_begin") + 1).alias("span_bp")
)
repeats.head()


Now let's summarize total repeat content by class, and see what fraction of the genome each represents:

In [ ]:
by_class = (
    repeats.group_by("repeat_class")
    .agg(polars.col("span_bp").sum().alias("total_bp"))
    .sort("total_bp", descending=True)
)

genome_size_bp = s3.head_object(Bucket=bucket, Key=example["genome_pri"])["ContentLength"]  # approximation via file size
by_class = by_class.with_columns(
    (polars.col("total_bp") / genome_size_bp * 100).alias("percent_of_genome")
)
by_class


In [ ]:
fig, ax = plt.subplots(figsize=(10, 6), dpi=100, facecolor='white')
ax.barh(by_class["repeat_class"], by_class["percent_of_genome"], color="#3498db", edgecolor='white')
ax.set_xlabel("% of genome", fontsize=12, labelpad=10)
ax.set_title(f"Repeat content by class - {example['common_name']} ({example['genome_id']})",
             fontsize=14, pad=20, fontweight='bold')
ax.set_facecolor('#f8f9fa')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.show()


### Q: What is one unanswered question that you think could be answered using these data? Do you have any recommendations or advice for someone wanting to answer this question?

We'd be excited to see the community pursue **comparative functional variant interpretation**: using the whole-genome alignments (MAF/HAL) spanning ~70 million years of primate evolution together with the per-species variant calls, can positions that are variable in humans but conserved across most of the ~40 primate species in this collection be used to flag candidate human-lineage-specific regulatory or coding changes?

Our recommendation for getting started: begin with a single, well-studied locus rather than a genome-wide scan, use the `comparative/` prefix's pairwise MAF files (one per species vs. human) rather than the full multi-species HAL to keep the initial scope manageable, and cross-reference candidate positions against the per-sample `variants/` VCFs via each sample's `genome_id`.

We'd love to see what you build - open an issue or discussion on [our GitHub repository](https://github.com/sudmantlab/PanPrimate-T2T) to share your approach or results.


## Further resources

- Full dataset documentation: https://github.com/sudmantlab/PanPrimate-T2T/blob/main/README_dataset.md
- Step-by-step AWS access guide: https://github.com/sudmantlab/PanPrimate-T2T/blob/main/tutorials/getting_started_on_aws.md
- Species-level browsing (photos, IUCN status, assembly status): https://github.com/sudmantlab/PanPrimate-T2T/blob/main/species/README.md
- Registry of Open Data listing: https://registry.opendata.aws/panprimate-t2t
